In [23]:
from pretokenization_example import find_chunk_boundaries
import regex as re
import multiprocessing
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


In [24]:
def initialize_vocab(special_tokens: list[str]):
    """Initialize the vocabulary with single-byte tokens and special tokens."""

    vocab: dict[int, bytes] = {}
    for i in range(256):
        vocab[i] = bytes([i])
    vocab[256] = b"<|endoftext|>"
    return vocab


In [25]:

def count_pretokens(matches) -> dict[str, int]:
    """Count the occurrences of each pretoken in the matches."""
    counts: dict[str, int] = {}
    for match in matches:
        pretoken = match.group(0)
        counts[pretoken] = counts.get(pretoken, 0) + 1
    return counts


In [ ]:

def update_pretoken_counts(pretoken_counts: dict[str, int], local_counts: dict[str, int]):
    """Update the global pretoken counts with the local counts from a chunk."""
    for key in local_counts.keys():
        pretoken_counts[key] = pretoken_counts.get(key, 0) + local_counts[key]

def encode_pretoken_counts(pretoken_counts: dict[str, int]):
    """Encode the pretoken counts dictionary to use bytes as keys."""

    pretoken_sequcence_to_counts: dict[tuple[bytes, ...], int] = {}
    for key in pretoken_counts.keys():
        encoded_key = key.encode("utf-8")
        encoded_key_sequence = tuple(encoded_key[i: i + 1] for i in range(len(encoded_key)))
        cnt = pretoken_counts[key]
        pretoken_sequcence_to_counts[encoded_key_sequence] = cnt
    return pretoken_sequcence_to_counts


In [ ]:
def count_token_pairs(pretoken_sequcence_to_counts: dict[tuple[bytes, ...], int]):
    """Count the occurrences of each pair of tokens in the pretoken counts."""
    pair_counts: dict[tuple[bytes, bytes], int] = {}
    pair_to_sequence: dict[tuple[bytes, bytes], list[tuple[bytes, ...]]] = {}

    for pretoken, count in pretoken_sequcence_to_counts.items():
        len = pretoken.__len__()
        for i in range(len - 1):
            pair = (pretoken[i : i + 1], pretoken[i + 1 : i + 2])
            pair_counts[pair] = pair_counts.get(pair, 0) + count
            pair_to_sequence.setdefault(pair, []).append(pretoken)

    return pair_counts, pair_to_sequence

In [ ]:
def merge_pretoken_pairs(pair_counts: dict[tuple[bytes, bytes], int], pair_to_sequence: dict[tuple[bytes, bytes], list[tuple[bytes, ...]]], pretoken_sequcence_to_counts: dict[tuple[bytes, ...], int], vocab: dict[int, bytes], merges: list[tuple[bytes, bytes]], vocab_size: int):
    """Merge the specified pair of tokens in the pretoken counts."""
    
    pair_to_merge = vocab_size - vocab.__len__()
    while pair_to_merge > 0:
        if not pair_counts:
            break

        # Find the most frequent pair
        most_frequent_pair = max(pair_counts, key=lambda key: (pair_counts[key], key))

        idx = vocab.__len__()
        vocab[idx] = most_frequent_pair[0] + most_frequent_pair[1]
        merges.append(most_frequent_pair)

        # Update the pretoken counts by merging the most frequent pair

        
        pair_to_merge -= 1

In [ ]:

def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]) -> dict[str, int]:
    """
    Train a BPE tokenizer on the given input file."""

    vocab: dict[int, bytes] = initialize_vocab(special_tokens)
    merges: list[tuple[bytes, bytes]] = []
    pretoken_counts: dict[str, int] = {}

    with open(input_path, "rb") as f:
        num_processes = 4
        boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

        # The following is a serial implementation, but you can parallelize this
        # by sending each start/end pair to a set of processes.
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            f.seek(start)
            chunk = f.read(end - start).decode("utf-8", errors="ignore")
            parts = chunk.split("<|endoftext|>")
            for part in parts:

                matches = re.finditer(PAT, part)

                local_counts = count_pretokens(matches)
                update_pretoken_counts(pretoken_counts, local_counts)
    
    pretoken_sequcence_to_counts = encode_pretoken_counts(pretoken_counts)
    pair_counts, pair_to_sequence = count_token_pairs(pretoken_sequcence_to_counts)

    merge_pretoken_pairs(pair_counts, pair_to_sequence, pretoken_sequcence_to_counts, vocab, merges, vocab_size)
    return vocab, merges

In [30]:
file_path = "/Users/felix/project/CS336/assignments/assignment1-basics/data/test.txt"
train_bpe(file_path, vocab_size=10000, special_tokens=["<|endoftext|>"])

({0: b'\x00',
  1: b'\x01',
  2: b'\x02',
  3: b'\x03',
  4: b'\x04',
  5: b'\x05',
  6: b'\x06',
  7: b'\x07',
  8: b'\x08',
  9: b'\t',
  10: b'\n',
  11: b'\x0b',
  12: b'\x0c',
  13: b'\r',
  14: b'\x0e',
  15: b'\x0f',
  16: b'\x10',
  17: b'\x11',
  18: b'\x12',
  19: b'\x13',
  20: b'\x14',
  21: b'\x15',
  22: b'\x16',
  23: b'\x17',
  24: b'\x18',
  25: b'\x19',
  26: b'\x1a',
  27: b'\x1b',
  28: b'\x1c',
  29: b'\x1d',
  30: b'\x1e',
  31: b'\x1f',
  32: b' ',
  33: b'!',
  34: b'"',
  35: b'#',
  36: b'$',
  37: b'%',
  38: b'&',
  39: b"'",
  40: b'(',
  41: b')',
  42: b'*',
  43: b'+',
  44: b',',
  45: b'-',
  46: b'.',
  47: b'/',
  48: b'0',
  49: b'1',
  50: b'2',
  51: b'3',
  52: b'4',
  53: b'5',
  54: b'6',
  55: b'7',
  56: b'8',
  57: b'9',
  58: b':',
  59: b';',
  60: b'<',
  61: b'=',
  62: b'>',
  63: b'?',
  64: b'@',
  65: b'A',
  66: b'B',
  67: b'C',
  68: b'D',
  69: b'E',
  70: b'F',
  71: b'G',
  72: b'H',
  73: b'I',
  74: b'J',
  75: b'K',
  76: b'